In [ ]:
#| default_exp core

# Core

> Authentication, compliance profiles, resource groups, and the `GenAIStack` convenience class.

In [ ]:
#| export
import os
import json
import boto3

## Compliance profiles

> Plain dicts — pass as `**HIPAA`, `**ISO27001`, or `**SOC2` to any `create_*` function.

In [ ]:
#| export
HIPAA = dict(
    encryption=True, tls_min='1.2', audit=True, multi_az=True,
    backup_retention=35, deletion_protection=True,
    tags={'compliance': 'hipaa'},
)

ISO27001 = dict(
    encryption=True, audit=True, managed_role=True,
    least_privilege=True, tls_min='1.2',
    tags={'compliance': 'iso27001'},
)

SOC2 = dict(
    encryption=True, audit=True, mfa_required=True, backup_retention=7,
    tags={'compliance': 'soc2'},
)

## AWSAuth

In [ ]:
#| export
class AWSAuth:
    'boto3.Session with credential chain. Reads AWS_DEFAULT_REGION from env.'
    def __init__(self, region=None, profile=None, role_arn=None):
        self.session = boto3.Session(
            profile_name=profile,
            region_name=region or os.environ.get('AWS_DEFAULT_REGION', 'us-east-1'))
        if role_arn:
            creds = self.session.client('sts').assume_role(
                RoleArn=role_arn, RoleSessionName='awseasy')['Credentials']
            self.session = boto3.Session(
                aws_access_key_id=creds['AccessKeyId'],
                aws_secret_access_key=creds['SecretAccessKey'],
                aws_session_token=creds['SessionToken'],
                region_name=self.session.region_name)
        self.region = self.session.region_name
        self.account_id = self.session.client('sts').get_caller_identity()['Account']

## Resource Groups

In [ ]:
#| export
def _rg_client(auth):
    return auth.session.client('resource-groups')

def resource_group(auth, name, tags=None) -> dict:
    'Create or update an AWS Resource Group with tag-based query. Idempotent.'
    tags = tags or {}
    tag_filters = [{'Key': k, 'Values': [v]} for k, v in tags.items()] if tags else [
        {'Key': 'ResourceGroup', 'Values': [name]}]
    query = json.dumps({'ResourceTypeFilters': ['AWS::AllSupported'],
                        'TagFilters': tag_filters})
    client = _rg_client(auth)
    try:
        return client.update_group(GroupName=name,
                                   ResourceQuery={'Type': 'TAG_FILTERS_1_0',
                                                  'Query': query})['Group']
    except client.exceptions.NotFoundException:
        return client.create_group(
            Name=name,
            ResourceQuery={'Type': 'TAG_FILTERS_1_0', 'Query': query},
            Tags={**tags, 'ResourceGroup': name},
        )['Group']

def list_resource_groups(auth) -> list:
    'List all resource groups in the account/region.'
    paginator = _rg_client(auth).get_paginator('list_groups')
    return [g for page in paginator.paginate() for g in page['GroupIdentifiers']]

def delete_resource_group(auth, name):
    'Delete a resource group (does not delete underlying resources).'
    _rg_client(auth).delete_group(GroupName=name)

## GenAIStack

> Provision a full enterprise GenAI stack on AWS in one call.

```python
auth = AWSAuth()
stack = GenAIStack(auth, 'myapp', compliance=HIPAA)
stack.provision()
print(stack.summary())
```

In [ ]:
#| export
class GenAIStack:
    'Provision a full enterprise GenAI stack on AWS in one call.'
    def __init__(self, auth, name, compliance=None):
        self.auth, self.name = auth, name
        self.compliance = compliance or ISO27001
        self._resources = {}

    def provision(self, bedrock=True, opensearch=True, s3=True, dynamodb=True,
                  redis=True, eks=False, secrets_manager=True) -> dict:
        'Create all resources, wire IAM roles, store secrets in Secrets Manager.'
        from .network import create_role, attach_policy, create_secret
        from .ai import create_opensearch, create_kb
        from .data import create_bucket, create_table, create_redis
        from .compute import create_eks

        c = self.compliance
        name = self.name
        auth = self.auth

        # IAM role first — everything else references it
        role = create_role(auth, f'{name}-role',
                           trust_policy=_bedrock_trust_policy(auth.account_id))
        attach_policy(auth, f'{name}-role',
                      'arn:aws:iam::aws:policy/AmazonBedrockFullAccess')
        self._resources['role'] = role

        if secrets_manager:
            secret = create_secret(auth, f'{name}/config', '{"stack": "' + name + '"}')
            self._resources['secret'] = secret

        if s3:
            bucket_name = f'{name}-{auth.account_id}-data'
            self._resources['s3'] = create_bucket(auth, bucket_name, **c)

        if opensearch:
            self._resources['opensearch'] = create_opensearch(
                auth, f'{name}-search', **c)

        if bedrock and s3:
            self._resources['kb'] = create_kb(
                auth, f'{name}-kb',
                bucket=f'{name}-{auth.account_id}-data',
                role_arn=role['Role']['Arn'])

        if dynamodb:
            self._resources['dynamodb'] = create_table(
                auth, f'{name}-table', partition_key='id', **c)

        if redis:
            self._resources['redis'] = create_redis(auth, f'{name}-redis', **c)

        if eks:
            self._resources['eks'] = create_eks(auth, f'{name}-eks', **c)

        return self._resources

    def summary(self) -> dict:
        'Return provisioned resource identifiers (no raw secrets).'
        return {k: _resource_id(v) for k, v in self._resources.items()}

def _bedrock_trust_policy(account_id) -> dict:
    return {
        'Version': '2012-10-17',
        'Statement': [{
            'Effect': 'Allow',
            'Principal': {'Service': 'bedrock.amazonaws.com'},
            'Action': 'sts:AssumeRole',
            'Condition': {'StringEquals': {'aws:SourceAccount': account_id}},
        }],
    }

def _resource_id(v):
    if isinstance(v, dict):
        for key in ('knowledgeBaseId', 'domainName', 'DBInstanceIdentifier',
                    'ReplicationGroupId', 'ClusterName', 'ARN', 'Arn',
                    'BucketName', 'Name', 'name'):
            if key in v: return v[key]
        # nested
        for key in ('Role', 'Group', 'Secret'):
            if key in v: return v[key].get('Arn', str(v))
    return str(v)